In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 2


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.147749327123165
Epoch 2/100, Loss: 2.4087895154953003
Epoch 3/100, Loss: 2.193440265953541
Epoch 4/100, Loss: 2.403915151953697
Epoch 5/100, Loss: 2.5157173722982407
Epoch 6/100, Loss: 2.2789569422602654
Epoch 7/100, Loss: 2.305187076330185
Epoch 8/100, Loss: 2.414927676320076
Epoch 9/100, Loss: 2.5809671580791473
Epoch 10/100, Loss: 2.6208155751228333
Epoch 11/100, Loss: 2.177956350147724
Epoch 12/100, Loss: 2.3877772465348244
Epoch 13/100, Loss: 2.1228640004992485
Epoch 14/100, Loss: 3.045818828046322
Epoch 15/100, Loss: 2.9479726552963257
Epoch 16/100, Loss: 2.516314558684826


Epoch 17/100, Loss: 3.1239450499415398
Epoch 18/100, Loss: 2.538935214281082
Epoch 19/100, Loss: 2.4380269944667816
Epoch 20/100, Loss: 2.368872694671154
Epoch 21/100, Loss: 2.5496943295001984
Epoch 22/100, Loss: 2.3887137174606323
Epoch 23/100, Loss: 2.2447522282600403
Epoch 24/100, Loss: 2.444135844707489
Epoch 25/100, Loss: 2.46483451128006
Epoch 26/100, Loss: 2.7844764664769173
Epoch 27/100, Loss: 2.530507802963257
Epoch 28/100, Loss: 2.5426403880119324
Epoch 29/100, Loss: 2.494924359023571
Epoch 30/100, Loss: 2.4162238612771034
Epoch 31/100, Loss: 2.5552395582199097
Epoch 32/100, Loss: 2.3321240544319153


Epoch 33/100, Loss: 2.3493190482258797
Epoch 34/100, Loss: 2.2264853790402412
Epoch 35/100, Loss: 2.3759646378457546
Epoch 36/100, Loss: 2.7457101196050644
Epoch 37/100, Loss: 2.3223207890987396
Epoch 38/100, Loss: 2.4946498423814774
Epoch 39/100, Loss: 2.216513507068157
Epoch 40/100, Loss: 2.2187631353735924
Epoch 41/100, Loss: 2.4319026321172714
Epoch 42/100, Loss: 2.5322042405605316
Epoch 43/100, Loss: 2.112500250339508
Epoch 44/100, Loss: 2.3747209310531616
Epoch 45/100, Loss: 2.368357352912426
Epoch 46/100, Loss: 2.5519075468182564
Epoch 47/100, Loss: 2.945165164768696


Epoch 48/100, Loss: 2.2935241609811783
Epoch 49/100, Loss: 2.2455827221274376
Epoch 50/100, Loss: 2.308924086391926
Epoch 51/100, Loss: 2.4245763570070267
Epoch 52/100, Loss: 2.383218601346016
Epoch 53/100, Loss: 2.20985134691
Epoch 54/100, Loss: 2.2958069816231728
Epoch 55/100, Loss: 2.369023822247982
Epoch 56/100, Loss: 2.398487150669098
Epoch 57/100, Loss: 2.416868209838867
Epoch 58/100, Loss: 2.24892071634531
Epoch 59/100, Loss: 2.4147263690829277
Epoch 60/100, Loss: 2.4118506759405136


Epoch 61/100, Loss: 2.6037214398384094
Epoch 62/100, Loss: 2.575387641787529
Epoch 63/100, Loss: 2.5572927370667458
Epoch 64/100, Loss: 2.253600537776947
Epoch 65/100, Loss: 2.5198887810111046
Epoch 66/100, Loss: 2.5602486208081245
Epoch 67/100, Loss: 2.461237385869026
Epoch 68/100, Loss: 2.444426454603672
Epoch 69/100, Loss: 2.474324733018875
Epoch 70/100, Loss: 2.2677334919571877
Epoch 71/100, Loss: 2.232625260949135
Epoch 72/100, Loss: 2.4961933717131615
Epoch 73/100, Loss: 2.4369896352291107
Epoch 74/100, Loss: 2.3756589218974113
Epoch 75/100, Loss: 2.2483744770288467
Epoch 76/100, Loss: 2.5645241364836693


...

Epoch 35/100, Loss: 1.756764505058527
Epoch 36/100, Loss: 2.3500542119145393
Epoch 37/100, Loss: 1.631834976375103
Epoch 38/100, Loss: 1.7707170471549034
Epoch 39/100, Loss: 1.74472576379776
Epoch 40/100, Loss: 1.6696427091956139
Epoch 41/100, Loss: 1.9398043230175972
Epoch 42/100, Loss: 1.907216690480709
Epoch 43/100, Loss: 1.8962179943919182
Epoch 44/100, Loss: 1.830005906522274
Epoch 45/100, Loss: 1.8679482638835907
Epoch 46/100, Loss: 1.740936018526554
Epoch 47/100, Loss: 1.9514999836683273
Epoch 48/100, Loss: 1.9345941059291363
Epoch 49/100, Loss: 1.7973622046411037
Epoch 50/100, Loss: 1.7811032868921757


Epoch 51/100, Loss: 1.8020205050706863
Epoch 52/100, Loss: 1.873071439564228
Epoch 53/100, Loss: 1.8789538331329823
Epoch 54/100, Loss: 1.7857509665191174
Epoch 55/100, Loss: 1.8519399426877499
Epoch 56/100, Loss: 1.9762152209877968
Epoch 57/100, Loss: 1.712748870253563
Epoch 58/100, Loss: 1.6706449761986732
Epoch 59/100, Loss: 2.0426502376794815
Epoch 60/100, Loss: 1.8082281947135925
Epoch 61/100, Loss: 1.7852383330464363
Epoch 62/100, Loss: 2.0032310634851456
Epoch 63/100, Loss: 1.969694696366787
Epoch 64/100, Loss: 1.9165821932256222


Epoch 65/100, Loss: 2.2047003395855427
Epoch 66/100, Loss: 1.6952616944909096
Epoch 67/100, Loss: 1.895364210009575
Epoch 68/100, Loss: 1.8994276449084282
Epoch 69/100, Loss: 1.7172924615442753
Epoch 70/100, Loss: 1.8522237576544285
Epoch 71/100, Loss: 1.8849355019629002
Epoch 72/100, Loss: 1.8061016872525215
Epoch 73/100, Loss: 1.871707834303379
Epoch 74/100, Loss: 1.7465487942099571
Epoch 75/100, Loss: 1.7860144227743149
Epoch 76/100, Loss: 1.8710075728595257
Epoch 77/100, Loss: 1.685319434851408
Epoch 78/100, Loss: 1.8454430513083935
Epoch 79/100, Loss: 1.8044353760778904
Epoch 80/100, Loss: 1.8174819126725197


Epoch 81/100, Loss: 1.8851303532719612
Epoch 82/100, Loss: 1.889739267528057
Epoch 83/100, Loss: 1.7390306517481804
Epoch 84/100, Loss: 1.674478717148304
Epoch 85/100, Loss: 1.7857228256762028
Epoch 86/100, Loss: 1.820204734802246
Epoch 87/100, Loss: 1.7014538645744324
Epoch 88/100, Loss: 1.8901257254183292
Epoch 89/100, Loss: 1.7947192192077637
Epoch 90/100, Loss: 1.872559692710638
Epoch 91/100, Loss: 1.8186496272683144
Epoch 92/100, Loss: 1.745573129504919
Epoch 93/100, Loss: 1.7490531131625175
Epoch 94/100, Loss: 1.8072412684559822
Epoch 95/100, Loss: 1.8174058124423027
Epoch 96/100, Loss: 1.7361905314028263
Epoch 97/100, Loss: 1.9045740216970444


Epoch 98/100, Loss: 1.8595040291547775
Epoch 99/100, Loss: 1.8009655810892582
Epoch 100/100, Loss: 2.002383593469858
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.6075
